## Setup

### Imports

In [1]:
import os

import polars as pl
import pandas as pd

import synapseclient

### Constants

In [2]:
# Jupyter server should be running from src/python
# assumes auth_token.txt exists in src/python as described in setup instructions
AUTH_TOKEN = os.path.abspath("auth_token.txt")

# cached metadata table, parquet format, on synapse 
META_CACHE_SYNID = "syn74775994"

# Jupyter server should be running from src/python
LIMMA_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_cache",
    "limma"
)

In [3]:
# ensure limma directory exists
os.makedirs(LIMMA_DIR, exist_ok=True)

### Synapse login

In [4]:
with open(AUTH_TOKEN, "r") as atf:
    SYN = synapseclient.login(authToken=atf.read())


UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.14.0) is available. Your version (4.11.0) can be upgraded by typing:
   pip install --upgrade synapseclient

Python Synapse Client version 4.14.0 release notes

https://python-docs.synapse.org/news/


Welcome, dylan.ross!



### Load metadata from cache

In [5]:
meta = pl.read_parquet(SYN.get(META_CACHE_SYNID).path)
meta

[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_56464/3955249378.py:1: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  meta = pl.read_parquet(SYN.get(META_CACHE_SYNID).path)



[syn74775994:metadata.parquet]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/261016_diffex_overview/diffex_overview/aml_outcome_disparities/analysis/mutations/_cache/metadata.parquet, skipping download.


Sample,Age,Sex,Race,Study,FLT3_ITD,NPM1
str,f64,str,str,str,bool,bool
"""11-00261""",74.0,"""Male""","""White""","""BeatAML""",true,false
"""11-00503""",54.0,"""Female""","""White""","""BeatAML""",true,true
"""11-00475""",65.0,"""Male""","""White""","""BeatAML""",true,true
"""12-00032""",70.0,"""Male""","""White""","""BeatAML""",true,false
"""11-00376""",49.0,"""Male""","""White""","""BeatAML""",true,true
…,…,…,…,…,…,…
"""16-01109-Bridge""",33.0,"""Male""","""Black""","""pilotStudy""",false,false
"""16-01191-Bridge""",65.0,"""Female""","""White""","""pilotStudy""",false,false
"""17-00025-Bridge""",39.0,"""Female""","""White""","""pilotStudy""",true,true


## set up for differential expression using `limma`

In [8]:
# ensure alignment of sample order from omics data and metadata
combined_samples = [c for c in combined.columns if c not in ["OmicsType", "Feature"]]
meta_samples = meta["Sample"].to_list()
assert combined_samples == meta_samples

In [9]:
# select out metadata needed to set up for limma
limma_meta = (
    meta
    .with_columns(
        pl.concat_str([
            pl.col("Race"),
            pl.lit("_NPM1"),
            pl.when("NPM1").then(pl.lit("+")).otherwise(pl.lit("-")),
            pl.lit("/FLT3-ITD"),
            pl.when("FLT3_ITD").then(pl.lit("+")).otherwise(pl.lit("-")),
        ])
        .alias("Group")
    )
    .select("Sample", "Group")
    .to_pandas()
    # polars DF → pandas DF
    .set_index("Sample")
)

limma_meta

,Group
Sample,
11-00261,White_NPM1-/FLT3-ITD+
11-00503,White_NPM1+/FLT3-ITD+
11-00475,White_NPM1+/FLT3-ITD+
12-00032,White_NPM1-/FLT3-ITD+
11-00376,White_NPM1+/FLT3-ITD+
...,...
16-01109-Bridge,Black_NPM1-/FLT3-ITD-
16-01191-Bridge,White_NPM1-/FLT3-ITD-
17-00025-Bridge,White_NPM1+/FLT3-ITD+


In [10]:
limma_design = (pd.get_dummies(limma_meta["Group"], dtype=float))

limma_design

,Black_NPM1+/FLT3-ITD+,Black_NPM1+/FLT3-ITD-,Black_NPM1-/FLT3-ITD+,Black_NPM1-/FLT3-ITD-,White_NPM1+/FLT3-ITD+,White_NPM1+/FLT3-ITD-,White_NPM1-/FLT3-ITD+,White_NPM1-/FLT3-ITD-
Sample,,,,,,,,
11-00261,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
11-00503,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
11-00475,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
12-00032,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
11-00376,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
16-01109-Bridge,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
16-01191-Bridge,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
17-00025-Bridge,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [11]:
limma_contrast = pd.DataFrame(
    0.,
    index=limma_design.columns.tolist(),
    columns=[
        # population comparison (-/-)
        "Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-",
        # NPM1+ vs NPM1-
        "Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-",
        "White_NPM1+/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-",
        # FLT3-ITD+ vs FLT3-ITD-
        "Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-",
        "White_NPM1-/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD-",
        # (+/+) vs NPM1+
        "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-",
        "White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-",
        # (+/+) vs FLT3-ITD+
        "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+",
        "White_NPM1+/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD+",
    ]
)

# define weights for contrasts
# population comparison (-/-)
limma_contrast.loc["Black_NPM1-/FLT3-ITD-", "Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-"] = 1.0
limma_contrast.loc["White_NPM1-/FLT3-ITD-", "Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-"] = -1.0
# NPM1+ vs NPM1-
limma_contrast.loc["Black_NPM1+/FLT3-ITD-", "Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-"] = 1.0
limma_contrast.loc["Black_NPM1-/FLT3-ITD-", "Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-"] = -1.0
limma_contrast.loc["White_NPM1+/FLT3-ITD-", "White_NPM1+/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-"] = 1.0
limma_contrast.loc["White_NPM1-/FLT3-ITD-", "White_NPM1+/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-"] = -1.0
# FLT3-ITD+ vs FLT3-ITD-
limma_contrast.loc["Black_NPM1-/FLT3-ITD+", "Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-"] = 1.0
limma_contrast.loc["Black_NPM1-/FLT3-ITD-", "Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-"] = -1.0
limma_contrast.loc["White_NPM1-/FLT3-ITD+", "White_NPM1-/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD-"] = 1.0
limma_contrast.loc["White_NPM1-/FLT3-ITD-", "White_NPM1-/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD-"] = -1.0
# (+/+) vs NPM1+
limma_contrast.loc["Black_NPM1+/FLT3-ITD+", "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-"] = 1.0
limma_contrast.loc["Black_NPM1+/FLT3-ITD-", "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-"] = -1.0
limma_contrast.loc["White_NPM1+/FLT3-ITD+", "White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-"] = 1.0
limma_contrast.loc["White_NPM1+/FLT3-ITD-", "White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-"] = -1.0
# (+/+) vs FLT3-ITD+
limma_contrast.loc["Black_NPM1+/FLT3-ITD+", "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+"] = 1.0
limma_contrast.loc["Black_NPM1-/FLT3-ITD+", "Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+"] = -1.0
limma_contrast.loc["White_NPM1+/FLT3-ITD+", "White_NPM1+/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD+"] = 1.0
limma_contrast.loc["White_NPM1-/FLT3-ITD+", "White_NPM1+/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD+"] = -1.0

limma_contrast

,Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-,Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-,White_NPM1+/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-,Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-,White_NPM1-/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD-,Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-,White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-,Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+,White_NPM1+/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD+
Black_NPM1+/FLT3-ITD+,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
Black_NPM1+/FLT3-ITD-,0.0,1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0
Black_NPM1-/FLT3-ITD+,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.0,0.0
Black_NPM1-/FLT3-ITD-,1.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
White_NPM1+/FLT3-ITD+,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
White_NPM1+/FLT3-ITD-,0.0,0.0,1.0,0.0,0.0,0.0,-1.0,0.0,0.0
White_NPM1-/FLT3-ITD+,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.0
White_NPM1-/FLT3-ITD-,-1.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0


In [13]:
# write all the limma inputs as parquet files
limma_design.to_parquet(os.path.join(LIMMA_DIR, "design.parquet"))
limma_contrast.to_parquet(os.path.join(LIMMA_DIR, "contrast.parquet"))